# 🔢 Aula 4 — Técnicas de contagem: maratonas de filmes e a Mega-Sena

Este notebook **não tem nota nem entrega**, e você **não precisa saber programar**. É só executar as células, na ordem, de cima para baixo, e ler os textos entre elas.

Na Aula 3, calculamos probabilidades **contando** os resultados do espaço amostral: casos favoráveis ÷ casos possíveis. Isso funciona bem quando dá para listar tudo (36 pares de dados). Mas e quando os casos são milhares ou milhões? A Aula 4 trouxe as ferramentas para **contar sem listar**: o **princípio multiplicativo**, o **arranjo** e a **combinação**. Aqui vamos usá-las com o catálogo de filmes e, no fim, com a **Mega-Sena**.

Cada seção segue sempre a mesma ordem:
1. **A situação e a pergunta** (o enunciado, como num exercício);
2. **A ferramenta da aula** que responde a pergunta;
3. **A resposta pela conta**;
4. **O que o código faz**, antes de cada célula;
5. **Como ler o resultado**.

> Notebook elaborado com auxílio de IA (Claude) e conferido pelo professor.

## Como executar

- Cada **célula de código** (as caixas cinza) tem um botão **▶** do lado esquerdo. Clique nele, ou clique na célula e aperte **Shift + Enter**.
- Execute **uma de cada vez, na ordem**: algumas células usam o resultado das anteriores.
- Na primeira vez, o Colab pode avisar que o notebook "não foi criado pelo Google". Clique em **Executar mesmo assim**.
- Se preferir, use o menu **Ambiente de execução → Executar tudo** e depois só leia, de cima para baixo.
- Detalhe: o Python escreve números com **ponto** no lugar da vírgula (`0.26` = 0,26).

## 0. Preparando as ferramentas e os dados

A célula abaixo carrega:

- **`math`**: tem as contas de contagem prontas: `math.perm(n, k)` (arranjo), `math.comb(n, k)` (combinação) e `math.factorial(n)` (fatorial, $n!$);
- **`itertools.combinations`**: **lista**, uma por uma, todas as combinações possíveis (útil para conferir as contas);
- **`numpy`**: faz sorteios em grande quantidade, para as simulações. A **semente 42** faz os sorteios saírem sempre iguais;
- **`pandas`**: trabalha com **tabelas**.

Depois, digita o catálogo de 24 filmes das aulas anteriores (cada lista entre colchetes é uma coluna).

> Os valores de nota e bilheteria são **aproximados**, usados só para fins didáticos.

In [ ]:
import math                              # arranjo, combinação, fatorial
from itertools import combinations       # lista todas as combinações
import numpy as np                       # sorteios em massa
import pandas as pd                      # tabelas

rng = np.random.default_rng(42)          # sorteador, semente 42

dados = {
    "titulo": ["Vingadores: Ultimato", "Toy Story", "O Poderoso Chefão", "Coringa",
               "Cidade de Deus", "Interestelar", "A Origem", "Shrek",
               "O Iluminado", "Parasita", "Bacurau", "Divertida Mente",
               "Duna", "Homem-Aranha: Sem Volta Para Casa", "Get Out - Corra!", "Klaus",
               "Tropa de Elite", "Zumbilândia", "Whiplash", "Coco",
               "Corra Que a Policia Vem Ai", "Não Olhe Para Cima", "Barbie", "Oppenheimer"],
    "genero": ["Ação", "Animação", "Drama", "Drama",
               "Drama", "Ficção Científica", "Ficção Científica", "Animação",
               "Terror", "Drama", "Ficção Científica", "Animação",
               "Ficção Científica", "Ação", "Terror", "Animação",
               "Ação", "Comédia", "Drama", "Animação",
               "Comédia", "Comédia", "Comédia", "Drama"],
    "classificacao_indicativa": ["12", "Livre", "16", "16",
                                  "18", "10", "12", "Livre",
                                  "16", "16", "16", "Livre",
                                  "12", "12", "16", "Livre",
                                  "18", "14", "12", "Livre",
                                  "12", "12", "10", "14"],
    "ano": [2019, 1995, 1972, 2019, 2002, 2014, 2010, 2001, 1980, 2019,
            2019, 2015, 2021, 2021, 2017, 2019, 2007, 2009, 2014, 2017,
            1988, 2021, 2023, 2023],
    "duracao_minutos": [181, 81, 175, 122, 130, 169, 148, 90, 146, 132,
                         131, 95, 155, 148, 104, 96, 115, 88, 107, 105,
                         85, 138, 114, 180],
    "nota_publico": [8.4, 8.3, 9.2, 8.4, 8.6, 8.7, 8.8, 7.9, 8.4, 8.6,
                      7.0, 8.1, 8.0, 8.3, 7.7, 7.5, 7.7, 7.6, 8.5, 8.4,
                      7.7, 7.2, 6.9, 8.4],
    "bilheteria_milhoes": [2797, 373, 246, 1074, 30, 701, 836, 484, 47, 258,
                            1.2, 858, 434, 1922, 255, 3.4, 25, 75, 13, 814,
                            36, 787, 1447, 950],
}

df = pd.DataFrame(dados)       # transforma o dicionário numa tabela
df.head(10)                    # mostra as 10 primeiras linhas

## 1. Quantas maratonas de fim de semana dá para montar?

### A situação e a pergunta
Você vai assistir **3 filmes diferentes** do catálogo: um na sexta, um no sábado e um no domingo. De quantas formas dá para montar essa programação?

### A ferramenta da aula
**Princípio multiplicativo:** se uma escolha tem $a$ opções e, depois dela, a próxima tem $b$ opções, as duas juntas têm $a \times b$ opções. Aqui a **ordem importa** (o filme de sexta é diferente do filme de domingo), então isso é um **arranjo**:

$$A(n, k) = \frac{n!}{(n-k)!}$$

### A resposta pela conta
Sexta: 24 opções; sábado: 23 (não repete); domingo: 22.

$$24 \times 23 \times 22 = A(24, 3) = 12.144$$

### O que o código faz
Faz a conta de dois jeitos: pelo princípio multiplicativo, direto, e com a função pronta de arranjo (`math.perm`).

In [ ]:
n = len(df)                                          # 24 filmes

multiplicativo = n * (n - 1) * (n - 2)               # sexta × sábado × domingo
arranjo = math.perm(n, 3)                            # A(24, 3) = 24! / 21!

print(f"Princípio multiplicativo: {n} × {n-1} × {n-2} = {multiplicativo}")
print(f"Arranjo A({n}, 3):        {arranjo}")

**Como ler:** os dois caminhos dão **12.144** programações. "Duna na sexta, Coco no sábado, Oppenheimer no domingo" conta como uma programação diferente de "Coco na sexta, Duna no sábado, Oppenheimer no domingo".

## 2. E se a ordem não importar?

### A pergunta
Agora você só quer escolher **quais** 3 filmes vai ver no fim de semana, em qualquer ordem. Quantos **grupos** de 3 filmes existem?

### A ferramenta da aula
Quando a ordem **não** importa, usamos a **combinação**:

$$C(n, k) = \binom{n}{k} = \frac{n!}{k!\,(n-k)!}$$

Cada grupo de 3 filmes aparece $3! = 6$ vezes entre os arranjos (uma para cada ordem), por isso $C(n,k) = A(n,k) / k!$.

### A resposta pela conta
$$\binom{24}{3} = \frac{12.144}{6} = 2.024$$

### O que o código faz
Calcula a combinação com `math.comb` e confere que ela é igual ao arranjo dividido por $3!$.

In [ ]:
combinacao = math.comb(n, 3)                          # C(24, 3)

print(f"Combinação C({n}, 3):       {combinacao}")
print(f"Arranjo ÷ 3!: {arranjo} ÷ {math.factorial(3)} = {arranjo // math.factorial(3)}")

**Como ler:** existem **2.024 grupos** de 3 filmes, seis vezes menos que as programações da seção 1: cada grupo corresponde a 6 programações diferentes (as 6 ordens possíveis dos mesmos 3 filmes).

## 3. Contagem a serviço da probabilidade: sorteando 3 filmes

### A situação e a pergunta
Para decidir o fim de semana, você **sorteia 3 filmes diferentes** do catálogo, todos com a mesma chance.
1. Qual a probabilidade de os **3 serem dramas**?
2. Qual a probabilidade de sair **exatamente 1 animação**?

### A ferramenta da aula
Como todos os grupos de 3 são igualmente prováveis:

$$P(\text{evento}) = \frac{\text{nº de grupos favoráveis}}{\text{nº total de grupos}} = \frac{\text{casos favoráveis}}{\binom{24}{3}}$$

e contamos os casos favoráveis com combinações e o princípio multiplicativo.

### A resposta pela conta
O catálogo tem **6 dramas** e **5 animações** (logo, 19 filmes que não são animação).

1. Escolher 3 entre os 6 dramas: $\binom{6}{3} = 20$ grupos. Então $P = \dfrac{20}{2.024} \approx 0{,}0099$ (cerca de 1%).
2. Escolher 1 das 5 animações **e** 2 entre os 19 outros filmes: $\binom{5}{1} \times \binom{19}{2} = 5 \times 171 = 855$. Então $P = \dfrac{855}{2.024} \approx 0{,}4224$.

### O que o código faz
1. conta quantos dramas e animações há no catálogo;
2. aplica as fórmulas acima;
3. **simula** 200.000 sorteios de 3 filmes (para cada sorteio, embaralha os 24 filmes e pega os 3 primeiros) e calcula em que fração deles cada evento aconteceu.

In [ ]:
n_drama = (df["genero"] == "Drama").sum()
n_anim = (df["genero"] == "Animação").sum()
total = math.comb(n, 3)

p_3_dramas = math.comb(n_drama, 3) / total                                  # C(6,3) / C(24,3)
p_1_anim = math.comb(n_anim, 1) * math.comb(n - n_anim, 2) / total          # C(5,1)·C(19,2) / C(24,3)

# simulação: 200.000 sorteios de 3 filmes diferentes
generos = df["genero"].to_numpy()
n_sorteios = 200_000
sorteados = np.argsort(rng.random((n_sorteios, n)), axis=1)[:, :3]         # embaralha e pega os 3 primeiros
generos_sorteados = generos[sorteados]                                     # o gênero de cada filme sorteado

emp_3_dramas = (generos_sorteados == "Drama").all(axis=1).mean()                # os 3 são drama
emp_1_anim = ((generos_sorteados == "Animação").sum(axis=1) == 1).mean()        # exatamente 1 animação

print(f"Dramas: {n_drama}   Animações: {n_anim}\n")
print(f"P(3 dramas)        teórico: {p_3_dramas:.4f}   empírico: {emp_3_dramas:.4f}")
print(f"P(1 animação)      teórico: {p_1_anim:.4f}   empírico: {emp_1_anim:.4f}")

**Como ler:** a chance de um fim de semana só de dramas é de **cerca de 1%**, e a de exatamente uma animação é de **cerca de 42%**. A simulação concorda com as contas (0,0101 contra 0,0099; 0,421 contra 0,422). As fórmulas de contagem deram a resposta **sem listar** os 2.024 grupos.

### Conferindo por "força bruta"
Com números pequenos, dá para **listar** todos os grupos e confirmar a conta. O código abaixo usa `combinations` para listar todos os trios de dramas possíveis, um por um.

In [ ]:
dramas = df[df["genero"] == "Drama"]["titulo"]
trios = list(combinations(dramas, 3))          # lista todos os grupos de 3 dramas

for trio in trios[:5]:                         # mostra só os 5 primeiros
    print(" + ".join(trio))
print("...")
print(f"Total de trios listados: {len(trios)}   |   C(6, 3) = {math.comb(6, 3)}")

**Como ler:** a lista completa tem **20** trios, exatamente $\binom{6}{3}$. A fórmula conta sem listar; a listagem só é viável porque aqui os números são pequenos.

## 4. Aplicação real: quais as chances na Mega-Sena?

### A situação
Na **Mega-Sena**, o apostador escolhe **6 números de 1 a 60**, e são sorteados 6. A ordem não importa. Com uma aposta simples:
- acerta a **sena** quem acerta os 6;
- a **quina**, quem acerta 5;
- a **quadra**, quem acerta 4.

### A pergunta
Qual a probabilidade de cada prêmio? E, jogando **uma vez por semana**, quanto tempo seria preciso, em média, para ganhar a sena?

### A ferramenta da aula
**Combinação** para contar os resultados possíveis e os favoráveis, e o **princípio multiplicativo** para juntar escolhas feitas ao mesmo tempo.

### A resposta pela conta
- Resultados possíveis: $\binom{60}{6} = 50.063.860$.
- **Sena:** só 1 resultado. $P = 1/50.063.860$.
- **Quina:** acertar 5 dos seus 6 números **e** 1 dos 54 que você não marcou: $\binom{6}{5} \times \binom{54}{1} = 6 \times 54 = 324$ resultados.
- **Quadra:** $\binom{6}{4} \times \binom{54}{2} = 15 \times 1.431 = 21.465$ resultados.

### O que o código faz
Aplica essas fórmulas, mostra cada chance também como "1 em quantos" e calcula quantos anos de apostas semanais equivalem a 50 milhões de jogos (52 semanas por ano).

In [ ]:
def br(x):
    return f"{x:,.0f}".replace(",", ".")               # escreve 50063860 como 50.063.860

possiveis = math.comb(60, 6)

premios = {
    "Sena":   math.comb(6, 6) * math.comb(54, 0),      # acerta 6
    "Quina":  math.comb(6, 5) * math.comb(54, 1),      # acerta 5 e erra 1
    "Quadra": math.comb(6, 4) * math.comb(54, 2),      # acerta 4 e erra 2
}

print(f"Resultados possíveis: C(60, 6) = {br(possiveis)}")
for nome, favoraveis in premios.items():
    print(f"{nome:<7} {favoraveis:>6} casos   P = {favoraveis / possiveis:.2e}   (1 em {br(possiveis / favoraveis)})")

print()
print(f"Apostando 1 vez por semana: {br(possiveis / 52)} anos, em média, para acertar a sena")

**Como ler:**
- A sena tem chance de **1 em 50.063.860** (a função `br` só escreve os números grandes com ponto de milhar, como no Brasil; `2.00e-08` é a notação científica do Python para $2{,}00 \times 10^{-8}$, ou 0,00000002).
- A quina sai 1 vez em cerca de **154 mil** jogos; a quadra, 1 vez em cerca de **2.332**.
- Jogando toda semana, levaria em média **cerca de 963 mil anos** para acertar a sena.

Esses números são as mesmas contas de combinação que fizemos com os filmes, só que com um espaço amostral de 50 milhões de resultados, impossível de listar à mão.

## Para fechar

Neste notebook usamos as ferramentas da Aula 4:

- **Princípio multiplicativo:** escolhas em sequência se multiplicam;
- **Arranjo:** quando a ordem importa ($A(n,k)$, `math.perm`);
- **Combinação:** quando a ordem não importa ($\binom{n}{k}$, `math.comb`);
- **Probabilidade por contagem:** casos favoráveis ÷ casos possíveis, contados com essas ferramentas, mesmo quando são milhões.

Na **Aula 5** a pergunta muda: como uma **informação nova** (por exemplo, "o teste deu positivo") altera uma probabilidade? É a **probabilidade condicional**.